# Youtube Scrapper

In [1]:
# STEP 1:Importing the necessary libraries
import chromedriver_binary 
from selenium import webdriver
from bs4 import BeautifulSoup
import time
import pandas as pd
from openpyxl import Workbook

In [3]:
# STEP 2:Setting up the WebDriver
driver = webdriver.Chrome()
driver.get("https://www.youtube.com/c/TechWithTim/videos")

The chromedriver version (147.0.7727.57) detected in PATH at c:\Users\Nbinary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\chromedriver_binary\chromedriver.exe might not be compatible with the detected chrome version (148.0.7778.168); currently, chromedriver 148.0.7778.178 is recommended for chrome 148.*, so it is advised to delete the driver in PATH and retry


In [4]:
# STEP 3:Scrolling to load more videos
last_height = driver.execute_script("return document.documentElement.scrollHeight")
while True:
    driver.execute_script("window.scrollTo(0, document.documentElement.scrollHeight);")
    time.sleep(2)  # Wait for the page to load
    new_height = driver.execute_script("return document.documentElement.scrollHeight")
    if new_height == last_height:
        break
    last_height = new_height

# Add extra wait to ensure page fully loads
time.sleep(3)

In [5]:
# STEP 4: Parsing the page source with BeautifulSoup
soup = BeautifulSoup(driver.page_source, 'html.parser')

# Try different selectors - modern YouTube uses these
video_elements = soup.find_all('ytd-rich-item-renderer')
print(f"Found {len(video_elements)} ytd-rich-item-renderer elements\n")

if video_elements:
    # Inspect the first element's structure
    first_video = video_elements[0]
    print("First video element structure:")
    print(f"Tag name: {first_video.name}")
    print(f"Direct children: {[child.name for child in first_video.children if hasattr(child, 'name') and child.name][:5]}")
    print(f"\nSearching for links in first element:")
    links = first_video.find_all('a')
    print(f"  - Found {len(links)} <a> tags")
    for j, link in enumerate(links[:3]):
        print(f"    Link {j}: id='{link.get('id')}', class='{link.get('class')}', href='{link.get('href')}'")
    
    print(f"\nSearching for images:")
    imgs = first_video.find_all('img')
    print(f"  - Found {len(imgs)} <img> tags")
    if imgs:
        print(f"    First image: src='{imgs[0].get('src')[:80]}...'")

Found 30 ytd-rich-item-renderer elements

First video element structure:
Tag name: ytd-rich-item-renderer
Direct children: ['div', 'yt-interaction']

Searching for links in first element:
  - Found 2 <a> tags
    Link 0: id='None', class='['ytLockupViewModelContentImage']', href='/watch?v=GFlFABWeqDc'
    Link 1: id='None', class='['ytLockupMetadataViewModelTitle']', href='/watch?v=GFlFABWeqDc'

Searching for images:
  - Found 1 <img> tags
    First image: src='https://i.ytimg.com/vi/GFlFABWeqDc/hqdefault.jpg?sqp=-oaymwEnCNACELwBSFryq4qpAxk...'


In [9]:
# STEP 5: Extracting video details

import re
from yt_dlp import YoutubeDL

videos = []

# yt-dlp config
ydl_opts = {
    "quiet": True,
    "no_warnings": True,
    "extract_flat": False
}

with YoutubeDL(ydl_opts) as ydl:

    for i, video in enumerate(video_elements):

        try:
            # Find title link
            title_link = video.find('a', {
                'class': 'ytLockupMetadataViewModelTitle'
            })

            if not title_link:
                continue

            # Extract title
            title = title_link.get(
                'aria-label',
                title_link.text.strip()
            )

            # Extract URL
            href = title_link.get('href', '')

            url = (
                f"https://www.youtube.com{href}"
                if href.startswith('/')
                else href
            )

            # -----------------------------
            # Fetch REAL metadata from YouTube
            # -----------------------------
            info = ydl.extract_info(url, download=False)

            views = info.get("view_count", 0)
            likes = info.get("like_count", 0)
            shares = info.get("repost_count", 0)

            # Optional extra fields
            uploader = info.get("uploader", "")
            duration = info.get("duration", 0)
            upload_date = info.get("upload_date", "")

            videos.append({
                'title': info.get("title", title),
                'url': url,
                'views': views,
                'likes': likes,
                'shares': shares,
                'channel': uploader,
                'duration_seconds': duration,
                'upload_date': upload_date
            })

            print(f"✓ Processed video {i+1}")

        except Exception as e:
            print(f"Error processing video {i}: {e}")

print(f"\n✓ Successfully extracted {len(videos)} videos\n")

✓ Processed video 1
✓ Processed video 2
✓ Processed video 3
✓ Processed video 4
✓ Processed video 5
✓ Processed video 6
✓ Processed video 7
✓ Processed video 8
✓ Processed video 9
✓ Processed video 10
✓ Processed video 11
✓ Processed video 12
✓ Processed video 13
✓ Processed video 14
✓ Processed video 15
✓ Processed video 16
✓ Processed video 17
✓ Processed video 18
✓ Processed video 19
✓ Processed video 20
✓ Processed video 21
✓ Processed video 22
✓ Processed video 23
✓ Processed video 24
✓ Processed video 25
✓ Processed video 26
✓ Processed video 27
✓ Processed video 28
✓ Processed video 29
✓ Processed video 30

✓ Successfully extracted 30 videos



In [10]:
# STEP 7: Deatils of all the vedios 
df = pd.DataFrame(videos)
print(df.head())

                                               title  \
0    Devin AI Is the Future of Coding… Full Tutorial   
1  AI Web Scraping Is Insanely Good | Browserbase...   
2  Claude Tutorial - How to Connect Claude to ANY...   
3  Claude Just Got a Superpower No One's Talking ...   
4                  One AI Agent Isn't Enough Anymore   

                                                 url  views  likes  shares  \
0        https://www.youtube.com/watch?v=GFlFABWeqDc   7270    194       0   
1        https://www.youtube.com/watch?v=XTQTJoSfeMg  11374    376       0   
2  https://www.youtube.com/watch?v=bzV2EwDyxpk&pp...  12219    324       0   
3  https://www.youtube.com/watch?v=rrylSizvnSg&pp...  35725    233       0   
4        https://www.youtube.com/watch?v=SVz-y-pD6s4  14187    231       0   

         channel  duration_seconds upload_date  
0  Tech With Tim              2286    20260518  
1  Tech With Tim              1256    20260515  
2  Tech With Tim               767    20260512 

In [11]:
# STEP 6: CREATE A DATEFRAME AND EXPORT TO CSV
# create a dataframe from the list of videos
df=pd.DataFrame(videos)
# save datafram to xslx file
df.to_excel('youtube_videos.xlsx', index=False)
print("✓ Data exported to youtube_videos.xlsx")

✓ Data exported to youtube_videos.xlsx
